In [73]:
import torch
import torch.nn as nn
import json
import re
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

with open("../../backend/data/processed/train.json", "r", encoding='utf-8') as f:
    train = json.load(f)
with open('../../backend/data/processed/validation.json', 'r', encoding = 'utf-8') as f:
    validation = json.load(f)
with open('../../backend/data/processed/test.json', 'r', encoding='utf-8') as f:
    test = json.load(f)

print(len(train))
print(len(test))
print(len(validation))

print(train[100])
print(type(train))
print(type(train[0]))

9724
1071
695
{'text': 'Sáng kêu mệt mỏi, ấm đầu', 'label': 'health'}
<class 'list'>
<class 'dict'>


In [74]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[.!?,;:]', '', text)
    return text.split()

In [92]:
from collections import Counter
counter = Counter()
for item in train:
    tokens = tokenizer((item['text']))
    counter.update(tokens)
print(counter.most_common(100))
print(len(counter))

[('ăn', 3890), ('ngủ', 2543), ('chơi', 1655), ('hết', 1457), ('không', 1054), ('tự', 1019), ('xúc', 1014), ('một', 962), ('cơm', 867), ('ngoan', 784), ('tập', 744), ('trung', 744), ('lơ', 705), ('đễnh', 705), ('bạn', 703), ('trưa', 695), ('đồ', 677), ('ngon', 633), ('với', 629), ('khóc', 617), ('chịu', 595), ('động', 585), ('mới', 565), ('cực', 552), ('hào', 522), ('hứng', 522), ('tích', 508), ('cùng', 502), ('mình', 488), ('cô', 477), ('bỏ', 463), ('mứa', 463), ('hay', 451), ('bát', 441), ('bé', 428), ('đòi', 425), ('sữa', 413), ('năng', 410), ('phải', 409), ('cháo', 406), ('gần', 405), ('chán', 370), ('đau', 340), ('mạch', 331), ('nửa', 331), ('nhè', 329), ('nay', 323), ('ít', 323), ('ho', 318), ('giấc', 315), ('bánh', 311), ('khỏe', 301), ('mệt', 300), ('thịt', 294), ('canh', 289), ('ngắn', 287), ('mỏi', 283), ('ngậm', 282), ('dài', 280), ('trằn', 278), ('trọc', 278), ('xin', 277), ('khó', 272), ('bị', 264), ('hình', 263), ('thức', 260), ('rau', 259), ('sốt', 258), ('xếp', 255), ('h

In [76]:
max_vocab_size = 20000
vocab = {
    '<pad>':0,
    '<unk>':1
}
for word , counter in counter.most_common(max_vocab_size):
    vocab[word] = len(vocab)
print(len(vocab))

367


In [77]:
print("Number of samples:", len(train))
print("Vocab size:", len(vocab))
print(train[0])
print(train[1])
print(train[2])

Number of samples: 9724
Vocab size: 367
{'text': 'Nay trộm vía con đi học ngoan, tự xúc ăn hết bát cơm luôn', 'label': 'meals'}
{'text': ' Trưa nay bé ngủ một mạch, ngủ ngon lắm', 'label': 'sleeping'}
{'text': ' Người hoàn toàn khỏe mạnh', 'label': 'health'}


In [78]:
def numericalize(text):
    tokens = tokenizer(text)
    return[
        vocab.get(token, vocab[('<unk>')])
        for token in tokens
    ]
text = "trưa nay bé Kiệt lặc ăn không hết phần cơm, ngủ ít"
numericalized = numericalize(text)
print(numericalized)

[17, 48, 36, 1, 1, 2, 6, 5, 163, 10, 3, 49]


In [79]:
label_to_id = {
    'meals' : 0,
    'sleeping' : 1,
    'health' : 2,
    'activities' : 3,
    'mood' : 4
}

In [80]:
class kindergatern_dataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        item = self.dataset[index]
        text = numericalize(item['text'])
        label = label_to_id[item['label']]
        return torch.tensor(text), torch.tensor(label)
        

In [81]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.stack(labels)
    return texts, labels
train_dataset = kindergatern_dataset(train)
validation_dataset = kindergatern_dataset(validation)
test_dataset = kindergatern_dataset(test)
print(type(train_dataset))
print(type(train_dataset.dataset))
print(type(train_dataset.dataset[0]))
print(train_dataset.dataset[0])


<class '__main__.kindergatern_dataset'>
<class 'list'>
<class 'dict'>
{'text': 'Nay trộm vía con đi học ngoan, tự xúc ăn hết bát cơm luôn', 'label': 'meals'}


In [82]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    collate_fn = collate_fn
)

validation_dataloader = DataLoader(
    validation_dataset,
    batch_size = 32,
    shuffle = False,
    collate_fn = collate_fn
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False,
    collate_fn = collate_fn
)

In [83]:
texts, labels = next(iter(train_dataloader))
print(texts.shape)
print(labels.shape)
embedding_dim = 128

torch.nn.Embedding(
    max_vocab_size, embedding_dim
)

torch.Size([32, 14])
torch.Size([32])


Embedding(20000, 128)

In [84]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(). __init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx = 0
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        embedd = self.embedding(x)
        out_put, (hidden, cell) = self.lstm(embedd)
        hidden = hidden[-1]
        out_put = self.fc(hidden)
        return out_put

In [85]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = LSTMClassifier(
    vocab_size=max_vocab_size,
    embedding_dim = embedding_dim,
    hidden_dim = 64,
    num_classes = 5
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

cuda


In [90]:
num_epoch = 5
for epoch in range(num_epoch):
    model.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts = texts.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out_put = model(texts)
        loss = criterion(out_put, labels)
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()
    print(f"Epoch {epoch + 1}/{num_epoch}, Loss: {loss.item():.4f}")


Epoch 1/5, Loss: 0.0001
Epoch 2/5, Loss: 0.0000
Epoch 3/5, Loss: 0.0000
Epoch 4/5, Loss: 0.0000
Epoch 5/5, Loss: 0.0000


In [94]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_dataloader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.9832


In [95]:
train_texts = set(item["text"] for item in train)
validation_texts = set(item["text"] for item in validation)
test_texts = set(item["text"] for item in test)

print("Train ∩ Validation:", len(train_texts & validation_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(validation_texts & test_texts))

Train ∩ Validation: 28
Train ∩ Test: 211
Validation ∩ Test: 28
